In [ ]:
import subprocess, sys, signal, os, json, math, time, random, warnings
from pathlib import Path
from collections import defaultdict
from typing import Optional

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["albumentations", "scikit-learn", "seaborn"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        pip_install(pkg)

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib
matplotlib.use("Agg")          # non-interactive: safe on Kaggle / headless
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, matthews_corrcoef,
    roc_auc_score, roc_curve, auc,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")


In [ ]:
class Config:
    # ── Paths ──────────────────────────────────────────────────
    DATA_ROOT      = "/kaggle/input"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    LOG_DIR        = "/kaggle/working/logs"

    # ── Dataset ────────────────────────────────────────────────
    CLASSES     = ["Benign", "InSitu", "Invasive", "Normal"]
    NUM_CLASSES = 4
    VAL_SPLIT   = 0.20

    # ── Model ──────────────────────────────────────────────────
    DROPOUT = 0.45              # backed off: went from overfit->underfit
    USE_SE  = True

    # ── Training ───────────────────────────────────────────────
    EPOCHS             = 120
    BATCH_SIZE         = 1
    ACCUMULATION_STEPS = 16    # effective batch = 16
    NUM_WORKERS        = 2
    PIN_MEMORY         = True

    # ── Checkpointing ──────────────────────────────────────────
    # Save a "panic" checkpoint every N optimiser steps within an epoch.
    # Lets you resume even if Kaggle kills the kernel mid-epoch.
    INTRA_EPOCH_SAVE_STEPS = 50

    # ── Optimizer ──────────────────────────────────────────────
    LR           = 7e-4         # was 1e-3
    WEIGHT_DECAY = 5e-4         # was 1e-4

    # ── Scheduler ──────────────────────────────────────────────
    WARMUP_EPOCHS = 5
    MIN_LR        = 1e-6

    # ── Regularisation ─────────────────────────────────────────
    LABEL_SMOOTHING = 0.10       # backed off: over-regularised
    MIXUP_ALPHA     = 0.40
    MIX_PROB        = 0.00       # disabled: no-op with BATCH_SIZE=1 (randperm(1) always=[0])
    CUTMIX_ALPHA    = 1.00

    # ── AMP + early stopping ───────────────────────────────────
    USE_AMP  = True
    PATIENCE = 30
    SEED     = 42

    # ── Stochastic depth & class re-weighting ─────────────────────────
    STOCHASTIC_DEPTH_RATE = 0.10   # NEW: linearly distributed drop path
    KEEP_BEST_HISTORY     = True   # NEW: unique file per best epoch

    CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}


In [ ]:
# ─── Graceful Kaggle / SIGTERM Shutdown ──────────────────────────────────────
# Kaggle sends SIGTERM ~60 seconds before forcibly killing the kernel.
# Registering a handler lets the training loop detect this and save a
# "panic" checkpoint so the run can resume from that point.

_SHUTDOWN_REQUESTED = False

def _handle_signal(signum, frame):
    global _SHUTDOWN_REQUESTED
    print(f"\n\u26a0\ufe0f  Signal {signum} received — will checkpoint and exit cleanly.")
    _SHUTDOWN_REQUESTED = True

signal.signal(signal.SIGTERM, _handle_signal)
signal.signal(signal.SIGINT,  _handle_signal)


# ─── General Utilities ───────────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

def find_data_root() -> str:
    for root, dirs, _ in os.walk("/kaggle/input"):
        if all(c in dirs for c in ["Benign", "InSitu", "Invasive", "Normal"]):
            print(f"\u2705  Dataset at: {root}")
            return root
    raise FileNotFoundError("\u274c  Class folders not found under /kaggle/input")

class AverageMeter:
    def __init__(self):  self.reset()
    def reset(self):     self.val = self.avg = self.sum = self.count = 0
    def update(self, v, n=1):
        self.val   = v
        self.sum  += v * n
        self.count += n
        self.avg   = self.sum / self.count

def topk_accuracy(output: torch.Tensor, target: torch.Tensor, k: int = 1) -> float:
    with torch.no_grad():
        _, pred  = output.topk(k, 1, True, True)
        correct  = pred.t().eq(target.view(1, -1).expand_as(pred.t()))
        return correct[:k].reshape(-1).float().sum(0).item() * 100.0 / target.size(0)


In [ ]:
# ─── Depthwise Separable Conv ─────────────────────────────────────────────────
class DSConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1, dilation=1, bias=False):
        super().__init__()
        pad      = padding if dilation == 1 else dilation
        self.dw  = nn.Conv2d(in_ch, in_ch,  kernel, stride, pad, dilation=dilation, groups=in_ch, bias=bias)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
        nn.init.kaiming_normal_(self.dw.weight, mode="fan_out", nonlinearity="relu")
        nn.init.kaiming_normal_(self.pw.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x): return self.pw(self.dw(x))


# ─── Drop Path (Stochastic Depth) ─────────────────────────────────────────────
class DropPath(nn.Module):
    """Randomly drops the entire residual branch during training (keeps identity/skip).
    Rate distributed linearly across blocks: first block drop_prob≈0, last≈max_rate."""
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob     = 1.0 - self.drop_prob
        shape         = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor = torch.floor(random_tensor + keep_prob)
        return x * random_tensor / keep_prob


# ─── Squeeze-Excitation Block ─────────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        h = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, h, bias=False), nn.ReLU(inplace=True),
            nn.Linear(h, channels, bias=False), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)


# ─── Bottleneck Block ─────────────────────────────────────────────────────────
class DSBottleneck(nn.Module):
    expansion = 4
    def __init__(self, in_ch, mid_ch, stride=1, downsample=None, use_se=True, drop_path_rate=0.0):
        super().__init__()
        out_ch = mid_ch * self.expansion
        self.conv1 = nn.Conv2d(in_ch, mid_ch, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(mid_ch)
        self.conv2 = DSConv2d(mid_ch, mid_ch, stride=stride)
        self.bn2   = nn.BatchNorm2d(mid_ch)
        self.conv3 = nn.Conv2d(mid_ch, out_ch, 1, bias=False)
        self.bn3   = nn.BatchNorm2d(out_ch)
        self.relu       = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.se         = SEBlock(out_ch) if use_se else nn.Identity()
        self.drop_path  = DropPath(drop_path_rate) if drop_path_rate > 0.0 else nn.Identity()
        for m in [self.conv1, self.conv3]:
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = self.se(out)
        identity = self.downsample(x) if self.downsample is not None else x
        return self.relu(self.drop_path(out) + identity)


# ─── DSResNet50 ───────────────────────────────────────────────────────────────
class DSResNet50(nn.Module):
    def __init__(self, num_classes=4, dropout=0.5, use_se=True, stochastic_depth_rate=0.0):
        super().__init__()
        self.use_se  = use_se
        total_blocks = 16  # 3+4+6+3
        self._dp_rates = [stochastic_depth_rate * i / (total_blocks - 1) for i in range(total_blocks)]
        self._dp_idx   = 0  # cursor advances as _make_layer consumes rates
        self.stem    = nn.Sequential(
            # Regular Conv2d for stem: needs cross-channel RGB spatial patterns
            # (DSConv on 3 channels = 3 independent filters; no cross-colour learning)
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1  = self._make_layer(64,   64,  3, 1)
        self.layer2  = self._make_layer(256, 128,  4, 2)
        self.layer3  = self._make_layer(512, 256,  6, 2)
        self.layer4  = self._make_layer(1024,512,  3, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout)
        self.fc      = nn.Linear(2048, num_classes)
        nn.init.kaiming_normal_(self.stem[0].weight, mode="fan_out", nonlinearity="relu")
        nn.init.normal_(self.fc.weight, 0, 0.01)
        nn.init.zeros_(self.fc.bias)

    def _make_layer(self, in_ch, mid_ch, blocks, stride):
        out_ch = mid_ch * DSBottleneck.expansion
        ds     = None
        if stride != 1 or in_ch != out_ch:
            ds = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        # ── wire DropPath rates (were computed but never passed before) ──────
        layers = [DSBottleneck(in_ch, mid_ch, stride, ds, self.use_se,
                               drop_path_rate=self._dp_rates[self._dp_idx])]
        self._dp_idx += 1
        for _ in range(1, blocks):
            layers.append(DSBottleneck(out_ch, mid_ch, use_se=self.use_se,
                                       drop_path_rate=self._dp_rates[self._dp_idx]))
            self._dp_idx += 1
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(self.dropout(x))

    def param_count(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable


In [ ]:
def build_transforms(mode: str) -> A.Compose:
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    if mode == "train":
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.10, scale_limit=0.15, rotate_limit=45,
                               border_mode=cv2.BORDER_REFLECT_101, p=0.60),
            # ElasticTransform / GridDistortion / OpticalDistortion REMOVED:
            # At 2048px these warp cell boundaries and glandular architecture —
            # exactly the features that distinguish InSitu from Invasive.
            A.OneOf([
                A.RandomBrightnessContrast(0.30, 0.30, p=1.0),
                A.ColorJitter(0.30, 0.30, 0.30, 0.10, p=1.0),
                A.HueSaturationValue(20, 30, 20, p=1.0),
            ], p=0.80),
            A.OneOf([
                # GaussNoise upper limit lowered: 60.0 erased chromatin texture
                A.GaussNoise(var_limit=(5.0, 20.0), p=1.0),
                # ISONoise REMOVED: camera sensor noise not present in slide scanners
                A.MultiplicativeNoise(multiplier=(0.90, 1.10), per_channel=True, p=1.0),
            ], p=0.40),
            A.OneOf([
                # MotionBlur removed: not physically present in microscopy slides
                A.GaussianBlur(blur_limit=(3, 7), p=1.0),
                A.Sharpen(alpha=(0.20, 0.50), lightness=(0.60, 1.00), p=1.0),
            ], p=0.40),
            # GridDropout REMOVED: erases ~500x500px blocks of tissue at 2048px
            # CoarseDropout reduced: 3 holes max, 96px — still teaches occlusion
            #   robustness without destroying diagnostic regions
            A.CoarseDropout(max_holes=3, max_height=96, max_width=96,
                            min_holes=1, min_height=32, min_width=32,
                            fill_value=0, p=0.25),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    return A.Compose([A.Normalize(mean=mean, std=std), ToTensorV2()])


def build_tta_transforms():
    """8-fold TTA: 4 rotations x 2 flips."""
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    tta = []
    for flip in [None, A.HorizontalFlip(p=1.0)]:
        for rot in [0, 90, 180, 270]:
            ops = []
            if flip: ops.append(flip)
            if rot:  ops.append(A.Rotate(limit=(rot, rot), p=1.0))
            ops += [A.Normalize(mean=mean, std=std), ToTensorV2()]
            tta.append(A.Compose(ops))
    return tta


In [ ]:
IMAGE_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}

class BACHDataset(Dataset):
    CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

    def __init__(self, root, split="train", transform=None, val_split=0.20, seed=42):
        self.root      = Path(root)
        self.transform = transform
        self.c2i       = {c: i for i, c in enumerate(self.CLASSES)}
        all_samples    = []
        for cls in self.CLASSES:
            cls_dir = self.root / cls
            if not cls_dir.exists():
                print(f"\u26a0\ufe0f  Missing folder: {cls_dir}"); continue
            for f in sorted(cls_dir.iterdir()):
                if f.suffix.lower() in IMAGE_EXTS:
                    all_samples.append((str(f), self.c2i[cls]))
        if not all_samples:
            raise RuntimeError(f"No images found under {root}")

        rng      = random.Random(seed)
        by_class = defaultdict(list)
        for item in all_samples:
            by_class[item[1]].append(item)

        train_s, val_s = [], []
        for lbl, items in by_class.items():
            rng.shuffle(items)
            n_val = max(1, int(len(items) * val_split))
            val_s.extend(items[:n_val])
            train_s.extend(items[n_val:])

        self.samples = train_s if split == "train" else val_s
        self._print_stats(split)

    def _print_stats(self, split):
        counts = defaultdict(int)
        for _, lbl in self.samples: counts[self.CLASSES[lbl]] += 1
        print(f"\n\U0001f4c1  {split.upper()} \u2014 {len(self.samples)} images")
        for cls in self.CLASSES:
            print(f"    {cls:10s}: {counts[cls]}")

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None \
              else np.array(Image.open(path).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label

    def class_weights(self) -> torch.Tensor:
        counts = defaultdict(int)
        for _, lbl in self.samples: counts[lbl] += 1
        n = len(self.samples)
        w = torch.zeros(len(self.CLASSES))
        for i in range(len(self.CLASSES)):
            c    = counts.get(i, 0)
            w[i] = (n / (len(self.CLASSES) * c)) if c else 0.0
        return w


In [ ]:
def mixup(x, y, alpha, device):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx   = torch.randperm(x.size(0), device=device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def cutmix(x, y, alpha, device):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=device)
    B, C, H, W = x.shape
    r  = math.sqrt(1 - lam)
    cw, ch = int(W * r), int(H * r)
    cx, cy = random.randint(0, W), random.randint(0, H)
    x1, x2 = max(cx - cw // 2, 0), min(cx + cw // 2, W)
    y1, y2 = max(cy - ch // 2, 0), min(cy + ch // 2, H)
    mixed  = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam    = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed, y, y[idx], lam

def mixed_loss(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)


In [ ]:
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr=1e-3, min_lr=1e-6):
        self.opt           = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.base_lr       = base_lr
        self.min_lr        = min_lr
        self._epoch        = 0

    def _compute_lr(self):
        e = self._epoch
        if e < self.warmup_epochs:
            return self.base_lr * (e + 1) / self.warmup_epochs
        p = (e - self.warmup_epochs) / max(self.total_epochs - self.warmup_epochs, 1)
        return self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * p))

    def step(self):
        lr = self._compute_lr()
        for g in self.opt.param_groups: g["lr"] = lr
        self._epoch += 1
        return lr

    def state_dict(self):         return {"epoch": self._epoch}
    def load_state_dict(self, d): self._epoch = d["epoch"]


class CheckpointManager:
    """
    Saves checkpoints to CHECKPOINT_DIR:
      checkpoint_latest.pth         -- overwritten every epoch (primary resume point)
      checkpoint_best.pth           -- best validation accuracy ever
      checkpoint_epoch_NNNN.pth     -- permanent snapshot every 10 epochs
      checkpoint_panic.pth          -- mid-epoch save (intra-epoch / SIGTERM)

    Resume logic: load_latest() automatically picks whichever of
    checkpoint_latest and checkpoint_panic is more recent (higher epoch).
    """
    def __init__(self, ckpt_dir: str):
        self.ckpt_dir      = Path(ckpt_dir)
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.best_val_acc  = 0.0
        self.best_val_loss = float("inf")

    # ── end-of-epoch save ─────────────────────────────────────────────────────
    def save(self, state: dict, is_best: bool, epoch: int):
        p = self.ckpt_dir / "checkpoint_latest.pth"
        torch.save(state, p)
        print(f"   \U0001f4be  Latest checkpoint  -> {p.name}")

        if epoch % 10 == 0:
            periodic = self.ckpt_dir / f"checkpoint_epoch_{epoch:04d}.pth"
            torch.save(state, periodic)
            print(f"   \U0001f4be  Periodic snapshot  -> {periodic.name}")

        if is_best:
            # ── full training checkpoint ──────────────────────────────────────
            torch.save(state, self.ckpt_dir / "checkpoint_best.pth")
            print(f"   \U0001f3c6  Best checkpoint    -> checkpoint_best.pth")
            # ── unique per-epoch backup (never overwritten, survives Kaggle timeout) ──
            if state.get("config", {}).get("keep_best_history", True):
                bk = self.ckpt_dir / f"model_best_ep{epoch:04d}.pth"
                wt = {k: state[k] for k in ("model_state_dict",) if k in state}
                wt.update({k: state.get(k, None) for k in ("arch","num_classes","val_acc","epoch","classes")})
                torch.save(wt, bk)
                print(f"   \U0001f4be  Backup weights     -> {bk.name}")

            # ── weights-only file (portable, no optimizer state) ──────────────
            # This is the file to DOWNLOAD and use for inference / transfer learning.
            weights_only = {
                "model_state_dict": state["model_state_dict"],
                "val_acc":          state["val_acc"],
                "val_loss":         state["val_loss"],
                "epoch":            state["epoch"],
                "config":           state["config"],
                "classes":          ["Benign", "InSitu", "Invasive", "Normal"],
                "arch":             "DSResNet50",
            }
            wp = self.ckpt_dir / "model_weights_best.pth"
            torch.save(weights_only, wp)
            print(f"   \U0001f511  Weights-only file  -> {wp.name}  (download this for inference/study)")

    # ── panic / intra-epoch save ──────────────────────────────────────────────
    def panic_save(self, state: dict):
        p = self.ckpt_dir / "checkpoint_panic.pth"
        torch.save(state, p)
        print(f"   \U0001f6a8  Panic checkpoint   -> {p.name}")

    # ── resume ────────────────────────────────────────────────────────────────
    def load_latest(self, model, optimizer, scheduler, scaler, device="cpu"):
        """Pick the checkpoint with the highest saved epoch (latest OR panic)."""
        candidates = [
            self.ckpt_dir / "checkpoint_latest.pth",
            self.ckpt_dir / "checkpoint_panic.pth",
        ]
        best_path, best_epoch = None, -1
        for p in candidates:
            if p.exists():
                ck = torch.load(p, map_location="cpu", weights_only=False)
                ep = ck.get("epoch", -1)
                if ep > best_epoch:
                    best_epoch = ep
                    best_path  = p

        if best_path is None:
            print("\u2139\ufe0f   No checkpoint found \u2014 starting from scratch.")
            return 0, [], []

        print(f"\U0001f4c2  Resuming from {best_path.name}  (saved at epoch {best_epoch})")
        ck = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model_state_dict"])
        optimizer.load_state_dict(ck["optimizer_state_dict"])
        if scheduler and "scheduler_state_dict" in ck:
            scheduler.load_state_dict(ck["scheduler_state_dict"])
        if scaler and "scaler_state_dict" in ck:
            scaler.load_state_dict(ck["scaler_state_dict"])
        self.best_val_acc  = ck.get("best_val_acc",  0.0)
        self.best_val_loss = ck.get("best_val_loss", float("inf"))
        return ck["epoch"] + 1, ck.get("train_history", []), ck.get("val_history", [])

    def load_best(self, model, device="cpu"):
        p = self.ckpt_dir / "checkpoint_best.pth"
        if p.exists():
            model.load_state_dict(torch.load(p, map_location=device, weights_only=False)["model_state_dict"])
        return model


In [ ]:
# ─── Comprehensive Metrics ────────────────────────────────────────────────────
def compute_metrics(preds, labels, probs, class_names):
    """
    Returns dict with:
      classification_report  per-class precision / recall / F1 / support
      confusion_matrix       raw counts
      cohen_kappa            inter-rater agreement (-1 to 1)
      mcc                    Matthews Correlation Coefficient (-1 to 1)
      roc_auc_macro          macro AUC one-vs-rest
      roc_auc_weighted       weighted AUC one-vs-rest
    """
    m = {}
    m["classification_report"] = classification_report(
        labels, preds, target_names=class_names, output_dict=True, zero_division=0
    )
    m["confusion_matrix"] = confusion_matrix(labels, preds).tolist()
    m["cohen_kappa"]      = cohen_kappa_score(labels, preds)
    m["mcc"]              = matthews_corrcoef(labels, preds)

    if probs is not None and len(np.unique(labels)) == len(class_names):
        y_bin = label_binarize(labels, classes=list(range(len(class_names))))
        try:
            m["roc_auc_macro"]    = roc_auc_score(y_bin, probs, multi_class="ovr", average="macro")
            m["roc_auc_weighted"] = roc_auc_score(y_bin, probs, multi_class="ovr", average="weighted")
        except Exception:
            m["roc_auc_macro"] = m["roc_auc_weighted"] = None
    else:
        m["roc_auc_macro"] = m["roc_auc_weighted"] = None
    return m


def print_metrics(m, class_names):
    print("\n" + "=" * 62)
    print("  EVALUATION METRICS")
    print("=" * 62)
    rep = m["classification_report"]
    print(f"  {'Class':<12}  {'Precision':>9}  {'Recall':>9}  {'F1':>9}  {'Support':>8}")
    print("  " + "-" * 58)
    for cls in class_names:
        r = rep.get(cls, {})
        print(f"  {cls:<12}  {r.get('precision',0):>9.4f}  {r.get('recall',0):>9.4f}"
              f"  {r.get('f1-score',0):>9.4f}  {int(r.get('support',0)):>8d}")
    print("  " + "-" * 58)
    for avg in ["macro avg", "weighted avg"]:
        r = rep.get(avg, {})
        print(f"  {avg:<12}  {r.get('precision',0):>9.4f}  {r.get('recall',0):>9.4f}"
              f"  {r.get('f1-score',0):>9.4f}  {int(r.get('support',0)):>8d}")
    print()
    print(f"  Cohen Kappa     : {m['cohen_kappa']:.4f}")
    print(f"  MCC             : {m['mcc']:.4f}")
    if m["roc_auc_macro"] is not None:
        print(f"  ROC-AUC macro   : {m['roc_auc_macro']:.4f}")
        print(f"  ROC-AUC weighted: {m['roc_auc_weighted']:.4f}")
    print("=" * 62)


# ─── Plotting ─────────────────────────────────────────────────────────────────
def save_plots(t_hist, v_hist, log_dir, class_names=None, cm=None,
               probs=None, labels=None):
    """Save training curves, confusion matrix, and ROC curves as PNG files."""
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    if not t_hist:
        return

    epochs   = [h["epoch"] for h in t_hist]
    t_losses = [h["loss"]  for h in t_hist]
    v_losses = [h["loss"]  for h in v_hist]
    t_accs   = [h["acc"]   for h in t_hist]
    v_accs   = [h["acc"]   for h in v_hist]
    best_idx = int(np.argmax(v_accs))

    # ── training curves ───────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, t_losses, "b-o", ms=3, label="Train")
    ax1.plot(epochs, v_losses, "r-o", ms=3, label="Val")
    ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(True, alpha=0.4)

    ax2.plot(epochs, t_accs, "b-o", ms=3, label="Train")
    ax2.plot(epochs, v_accs, "r-o", ms=3, label="Val")
    ax2.axvline(epochs[best_idx], color="green", ls="--", alpha=0.7,
                label=f"Best {v_accs[best_idx]:.2f}% @ ep{epochs[best_idx]}")
    ax2.set_title("Accuracy (%)"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
    ax2.legend(); ax2.grid(True, alpha=0.4)

    plt.suptitle("DSResNet50 -- BACH Histology Training", fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = log_dir / "training_curves.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"   \U0001f4c8  Training curves    -> {out}")

    # ── confusion matrix ──────────────────────────────────────────────────────
    if cm is not None and class_names is not None:
        cm_arr  = np.array(cm)
        cm_norm = cm_arr.astype(float) / cm_arr.sum(axis=1, keepdims=True).clip(min=1)
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        for ax, data, fmt, title in zip(
            axes,
            [cm_arr, cm_norm],
            ["d",    ".2f"],
            ["Counts", "Row-normalised (Recall)"],
        ):
            sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                        xticklabels=class_names, yticklabels=class_names,
                        ax=ax, linewidths=0.5)
            ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        plt.suptitle("Confusion Matrix", fontsize=13, fontweight="bold")
        plt.tight_layout()
        out = log_dir / "confusion_matrix.png"
        plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
        print(f"   \U0001f4ca  Confusion matrix   -> {out}")

    # ── ROC curves ────────────────────────────────────────────────────────────
    if probs is not None and labels is not None and class_names is not None:
        y_bin   = label_binarize(labels, classes=list(range(len(class_names))))
        colors  = plt.cm.tab10.colors
        fig, ax = plt.subplots(figsize=(8, 6))
        for i, cls in enumerate(class_names):
            fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
            ax.plot(fpr, tpr, color=colors[i], lw=2,
                    label=f"{cls}  AUC={auc(fpr, tpr):.3f}")
        ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
        ax.set_title("ROC Curves (One-vs-Rest)")
        ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
        ax.legend(loc="lower right"); ax.grid(True, alpha=0.4)
        plt.tight_layout()
        out = log_dir / "roc_curves.png"
        plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
        print(f"   \U0001f4c9  ROC curves         -> {out}")


In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device, cfg,
                ckpt_mgr, epoch, scheduler, t_hist, v_hist):
    """
    Training loop with intra-epoch panic checkpoint and SIGTERM handler.
    Raises SystemExit on graceful shutdown so the caller can save final state.
    """
    global _SHUTDOWN_REQUESTED
    model.train()
    loss_m, acc_m = AverageMeter(), AverageMeter()
    optimizer.zero_grad()
    t0       = time.time()
    opt_step = 0

    for step, (imgs, lbls) in enumerate(loader):

        # ── graceful shutdown check ───────────────────────────────────────────
        if _SHUTDOWN_REQUESTED:
            ckpt_mgr.panic_save({
                "epoch":                epoch - 1,   # last *fully completed* epoch
                "model_state_dict":     model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict":    scaler.state_dict(),
                "best_val_acc":  ckpt_mgr.best_val_acc,
                "best_val_loss": ckpt_mgr.best_val_loss,
                "train_history": t_hist,
                "val_history":   v_hist,
                "note": f"SIGTERM mid-epoch {epoch} step {step}",
            })
            raise SystemExit("\u26d4  Graceful shutdown complete. Resume with load_latest().")

        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        if random.random() < cfg.MIX_PROB:
            # Mixed batch (80% of the time)
            if random.random() < 0.5:
                imgs, ya, yb, lam = mixup(imgs, lbls, cfg.MIXUP_ALPHA, device)
            else:
                imgs, ya, yb, lam = cutmix(imgs, lbls, cfg.CUTMIX_ALPHA, device)
        else:
            # Clean batch (20%) — hard single-label signal
            ya, yb, lam = lbls, lbls, 1.0

        with autocast("cuda", enabled=cfg.USE_AMP):
            out  = model(imgs)
            loss = mixed_loss(criterion, out, ya, yb, lam) / cfg.ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % cfg.ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            opt_step += 1

            # ── intra-epoch panic checkpoint ──────────────────────────────────
            if opt_step % cfg.INTRA_EPOCH_SAVE_STEPS == 0:
                ckpt_mgr.panic_save({
                    "epoch":                epoch - 1,
                    "model_state_dict":     model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict":    scaler.state_dict(),
                    "best_val_acc":  ckpt_mgr.best_val_acc,
                    "best_val_loss": ckpt_mgr.best_val_loss,
                    "train_history": t_hist,
                    "val_history":   v_hist,
                    "note": f"Intra-epoch: epoch {epoch}, opt_step {opt_step}",
                })

        # Track accuracy against primary label ya (lam-weighted dominant class).
        # lbls are original pre-mix indices — inaccurate against mixed output.
        with torch.no_grad():
            acc = topk_accuracy(out.detach(), ya)
        loss_m.update(loss.item() * cfg.ACCUMULATION_STEPS, imgs.size(0))
        acc_m.update(acc, imgs.size(0))

        if step % 5 == 0:
            print(f"\r   step {step+1:03d}/{len(loader)}"
                  f"  loss {loss_m.avg:.4f}  acc {acc_m.avg:.2f}%"
                  f"  ({time.time()-t0:.0f}s)", end="", flush=True)
    print()
    return loss_m.avg, acc_m.avg


@torch.no_grad()
def validate_epoch(model, loader, criterion, device, cfg):
    """Returns (loss, acc, preds, labels, probs)."""
    model.eval()
    loss_m, acc_m                    = AverageMeter(), AverageMeter()
    all_preds, all_labels, all_probs = [], [], []

    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        with autocast("cuda", enabled=cfg.USE_AMP):
            out  = model(imgs)
            loss = criterion(out, lbls)
        probs = torch.softmax(out, dim=1)
        acc   = topk_accuracy(out, lbls)
        loss_m.update(loss.item(), imgs.size(0))
        acc_m.update(acc, imgs.size(0))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return (loss_m.avg, acc_m.avg,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))


def predict_image(model, img_path, cfg, device, tta=True):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None \
          else np.array(Image.open(img_path).convert("RGB"))
    model.eval()
    transforms = build_tta_transforms() if tta else [build_transforms("val")]
    probs_list = []
    with torch.no_grad():
        for tfm in transforms:
            t = tfm(image=img)["image"].unsqueeze(0).to(device)
            with autocast("cuda", enabled=cfg.USE_AMP):
                p = torch.softmax(model(t), dim=1)
            probs_list.append(p)
    probs = torch.stack(probs_list).mean(0)
    pred  = probs.argmax(dim=1).item()
    return {
        "class":         cfg.CLASSES[pred],
        "confidence":    probs[0, pred].item(),
        "probabilities": {c: probs[0, i].item() for i, c in enumerate(cfg.CLASSES)},
    }


In [ ]:
def main():
    cfg = Config()
    set_seed(cfg.SEED)
    os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(cfg.LOG_DIR,        exist_ok=True)

    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data_root = find_data_root()

    # ── Datasets & loaders ────────────────────────────────────────────────────
    train_ds = BACHDataset(data_root, "train", build_transforms("train"),
                           val_split=cfg.VAL_SPLIT, seed=cfg.SEED)
    val_ds   = BACHDataset(data_root, "val",   build_transforms("val"),
                           val_split=cfg.VAL_SPLIT, seed=cfg.SEED)

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                              num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
                              drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False,
                              num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = DSResNet50(cfg.NUM_CLASSES, cfg.DROPOUT, cfg.USE_SE,
                       stochastic_depth_rate=cfg.STOCHASTIC_DEPTH_RATE).to(device)
    total, trainable = model.param_count()
    print(f"\n\U0001f9e0  Parameters: {total:,} total | {trainable:,} trainable")

    class_w   = train_ds.class_weights().to(device)
    
    print(f"   \u2696\ufe0f  Class weights: { {c: round(class_w[i].item(),3) for c,i in cfg.CLASS_TO_IDX.items()} }")
    criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=cfg.LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.LR,
                            weight_decay=cfg.WEIGHT_DECAY, betas=(0.9, 0.999))
    scheduler = WarmupCosineScheduler(optimizer, cfg.WARMUP_EPOCHS, cfg.EPOCHS,
                                      cfg.LR, cfg.MIN_LR)
    scaler    = GradScaler(enabled=cfg.USE_AMP)

    # ── Resume ────────────────────────────────────────────────────────────────
    ckpt_mgr = CheckpointManager(cfg.CHECKPOINT_DIR)
    start_epoch, t_hist, v_hist = ckpt_mgr.load_latest(
        model, optimizer, scheduler, scaler, device=str(device)
    )

    patience_counter = 0
    print(f"\n\U0001f680  Training from epoch {start_epoch + 1} / {cfg.EPOCHS}")
    print(f"   Panic save every {cfg.INTRA_EPOCH_SAVE_STEPS} optimiser steps (within epoch)")

    for epoch in range(start_epoch, cfg.EPOCHS):
        lr = scheduler.step()
        print(f"\n\U0001f4c5  Epoch {epoch+1:03d}/{cfg.EPOCHS}   lr={lr:.2e}")

        # ── Train ─────────────────────────────────────────────────────────────
        t_loss, t_acc = train_epoch(
            model, train_loader, criterion, optimizer, scaler, device, cfg,
            ckpt_mgr, epoch + 1, scheduler, t_hist, v_hist,
        )

        # ── Validate + metrics ────────────────────────────────────────────────
        v_loss, v_acc, v_preds, v_labels, v_probs = validate_epoch(
            model, val_loader, criterion, device, cfg
        )
        print(f"   \U0001f535 Train  loss={t_loss:.4f}  acc={t_acc:.2f}%")
        print(f"   \U0001f7e2 Val    loss={v_loss:.4f}  acc={v_acc:.2f}%")

        metrics = compute_metrics(v_preds, v_labels, v_probs, cfg.CLASSES)
        print_metrics(metrics, cfg.CLASSES)

        # ── History ───────────────────────────────────────────────────────────
        t_hist.append({"epoch": epoch+1, "loss": t_loss, "acc": t_acc, "lr": lr})
        v_hist.append({
            "epoch": epoch+1, "loss": v_loss, "acc": v_acc,
            "kappa": metrics["cohen_kappa"],
            "mcc":   metrics["mcc"],
            "auc":   metrics.get("roc_auc_macro"),
        })

        # ── Checkpoint ────────────────────────────────────────────────────────
        is_best = v_acc > ckpt_mgr.best_val_acc
        if is_best:
            ckpt_mgr.best_val_acc  = v_acc
            ckpt_mgr.best_val_loss = v_loss
            patience_counter       = 0
            print("   \U0001f3c6  New best val_acc!")
        else:
            patience_counter += 1

        ckpt_mgr.save(
            state={
                "epoch":                epoch,
                "model_state_dict":     model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict":    scaler.state_dict(),
                "train_loss": t_loss,   "train_acc": t_acc,
                "val_loss":   v_loss,   "val_acc":   v_acc,
                "best_val_acc":  ckpt_mgr.best_val_acc,
                "best_val_loss": ckpt_mgr.best_val_loss,
                "train_history": t_hist,
                "val_history":   v_hist,
                "config": {"num_classes": cfg.NUM_CLASSES, "batch_size": cfg.BATCH_SIZE},
            },
            is_best=is_best,
            epoch=epoch + 1,
        )

        # ── Save plots ────────────────────────────────────────────────────────
        save_plots(t_hist, v_hist, cfg.LOG_DIR,
                   class_names=cfg.CLASSES,
                   cm=metrics["confusion_matrix"],
                   probs=v_probs, labels=v_labels)

        # ── Early stopping ────────────────────────────────────────────────────
        if patience_counter >= cfg.PATIENCE:
            print(f"\n\u23f9\ufe0f  Early stopping after {cfg.PATIENCE} epochs without improvement.")
            break

    # ── Final evaluation on best weights ──────────────────────────────────────
    print("\n\U0001f504  Loading best weights for final evaluation ...")
    model = ckpt_mgr.load_best(model, device=str(device))
    _, best_acc, best_preds, best_labels, best_probs = validate_epoch(
        model, val_loader, criterion, device, cfg
    )
    print(f"   Final val_acc (best ckpt): {best_acc:.2f}%  |  best ever: {ckpt_mgr.best_val_acc:.2f}%")
    best_m = compute_metrics(best_preds, best_labels, best_probs, cfg.CLASSES)
    print_metrics(best_m, cfg.CLASSES)
    save_plots(t_hist, v_hist, cfg.LOG_DIR,
               class_names=cfg.CLASSES,
               cm=best_m["confusion_matrix"],
               probs=best_probs, labels=best_labels)

    # persist metrics as JSON
    metrics_path = Path(cfg.LOG_DIR) / "best_metrics.json"
    serialisable = {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in best_m.items()}
    with open(metrics_path, "w") as f:
        json.dump(serialisable, f, indent=2)
    print(f"\n\u2705  Training complete!  Plots + metrics -> {cfg.LOG_DIR}/")
    print(f"   \U0001f4c4  best_metrics.json  -> {metrics_path}")


if __name__ == "__main__":
    main()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  HOW THE WEIGHTS ARE SAVED — AND HOW TO USE THEM
# ═══════════════════════════════════════════════════════════════════════════════
#
#  After training, /kaggle/working/checkpoints/ will contain:
#
#    checkpoint_latest.pth       ← full training state (resume training here)
#    checkpoint_best.pth         ← full training state at best val_acc
#    checkpoint_epoch_NNNN.pth   ← periodic snapshots every 10 epochs
#    checkpoint_panic.pth        ← last mid-epoch panic save (if triggered)
#    model_weights_best.pth      ← WEIGHTS ONLY — download this for study/inference
#
#  Kaggle PERSISTS /kaggle/working between session restarts on the SAME notebook.
#  When you re-open the notebook and re-run, load_latest() automatically finds
#  and resumes from whichever checkpoint has the highest epoch.
#
#  To DOWNLOAD the weights: Output tab (right sidebar) → checkpoints/ → download.
# ───────────────────────────────────────────────────────────────────────────────

import os
from pathlib import Path

CKPT_DIR = "/kaggle/working/checkpoints"

def list_saved_weights(ckpt_dir=CKPT_DIR):
    """Show all saved files with sizes and best val_acc."""
    p = Path(ckpt_dir)
    if not p.exists():
        print("No checkpoint directory found yet — run training first.")
        return
    files = sorted(p.glob("*.pth"))
    if not files:
        print("No .pth files found yet."); return
    print(f"\n{'File':<40} {'Size':>10}  {'Epoch':>6}  {'Val Acc':>8}")
    print("-" * 70)
    for f in files:
        try:
            ck  = torch.load(f, map_location="cpu", weights_only=False)
            ep  = ck.get("epoch", "—")
            acc = ck.get("val_acc", ck.get("best_val_acc", "—"))
            acc_str = f"{acc:.2f}%" if isinstance(acc, float) else str(acc)
            size_mb = f.stat().st_size / 1e6
            print(f"  {f.name:<38} {size_mb:>8.1f}MB  {str(ep):>6}  {acc_str:>8}")
        except Exception:
            print(f"  {f.name:<38} (could not read)")
    print()

list_saved_weights()


## Load Weights for Inference or Further Study

Use the cell below to load `model_weights_best.pth` (weights-only, no optimizer state) for inference, visualisation, or transfer learning outside the training loop.

In [ ]:
# ─── Load weights for inference / further study ───────────────────────────────
# Works both inside Kaggle and after downloading the .pth file locally.

def load_model_for_inference(weights_path: str, device=None):
    """
    Load DSResNet50 from a weights-only checkpoint.
    Returns (model, metadata_dict).

    Usage:
        model, meta = load_model_for_inference("/kaggle/working/checkpoints/model_weights_best.pth")
        result = predict_image(model, "path/to/image.tif", cfg, device)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ck = torch.load(weights_path, map_location=device, weights_only=False)

    # Reconstruct model from saved config
    cfg_saved   = ck.get("config", {})
    num_classes = cfg_saved.get("num_classes", 4)

    model = DSResNet50(num_classes=num_classes, dropout=0.0, use_se=True)
    model.load_state_dict(ck["model_state_dict"])
    model.to(device)
    model.eval()                    # inference mode — BN uses running stats, dropout off

    meta = {
        "arch":       ck.get("arch",    "DSResNet50"),
        "classes":    ck.get("classes", ["Benign", "InSitu", "Invasive", "Normal"]),
        "val_acc":    ck.get("val_acc"),
        "val_loss":   ck.get("val_loss"),
        "epoch":      ck.get("epoch"),
    }

    print(f"\u2705  Model loaded from: {weights_path}")
    print(f"   Architecture : {meta['arch']}")
    print(f"   Classes      : {meta['classes']}")
    print(f"   Saved epoch  : {meta['epoch']}")
    if meta["val_acc"] is not None:
        print(f"   Val accuracy : {meta['val_acc']:.2f}%")
    return model, meta


# ─── Resume training from full checkpoint ────────────────────────────────────
def resume_training_checkpoint(full_ckpt_path: str, device=None):
    """
    Load a full training checkpoint (latest / best / panic).
    Returns model, optimizer, scheduler, scaler, start_epoch, t_hist, v_hist.

    Pass the returned objects straight into main()'s training loop.
    Useful for resuming from a specific epoch rather than the latest one.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    cfg = Config()
    ck  = torch.load(full_ckpt_path, map_location=device, weights_only=False)

    model     = DSResNet50(cfg.NUM_CLASSES, cfg.DROPOUT, cfg.USE_SE).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = WarmupCosineScheduler(optimizer, cfg.WARMUP_EPOCHS, cfg.EPOCHS, cfg.LR, cfg.MIN_LR)
    scaler    = GradScaler(enabled=cfg.USE_AMP)

    model.load_state_dict(ck["model_state_dict"])
    optimizer.load_state_dict(ck["optimizer_state_dict"])
    if "scheduler_state_dict" in ck:
        scheduler.load_state_dict(ck["scheduler_state_dict"])
    if "scaler_state_dict" in ck:
        scaler.load_state_dict(ck["scaler_state_dict"])

    start_epoch = ck["epoch"] + 1
    t_hist      = ck.get("train_history", [])
    v_hist      = ck.get("val_history",   [])

    print(f"\u2705  Full checkpoint loaded: {full_ckpt_path}")
    print(f"   Resuming from epoch {start_epoch}  |  best val_acc so far: {ck.get('best_val_acc', 0):.2f}%")
    return model, optimizer, scheduler, scaler, start_epoch, t_hist, v_hist


# ─── Export to ONNX (optional — for deployment / cross-framework study) ───────
def export_onnx(weights_path: str, out_path: str = "/kaggle/working/model_best.onnx",
                input_h: int = 512, input_w: int = 512):
    """
    Export model to ONNX for use in TensorRT, OpenCV DNN, or any ONNX runtime.
    input_h / input_w: dummy input spatial size (does not constrain inference size).
    """
    device = torch.device("cpu")
    model, _ = load_model_for_inference(weights_path, device=device)
    dummy = torch.randn(1, 3, input_h, input_w, device=device)
    torch.onnx.export(
        model, dummy, out_path,
        input_names=["image"],
        output_names=["logits"],
        dynamic_axes={"image": {0: "batch", 2: "height", 3: "width"},
                      "logits": {0: "batch"}},
        opset_version=17,
    )
    print(f"\u2705  ONNX model saved -> {out_path}")
    print(f"   Load with: import onnxruntime as ort; ort.InferenceSession(\"{out_path}\")")


# ─── Quick demo — run inference on the first val image ───────────────────────
WEIGHTS = "/kaggle/working/checkpoints/model_weights_best.pth"

if Path(WEIGHTS).exists():
    model_inf, meta = load_model_for_inference(WEIGHTS)
    cfg_inf         = Config()
    device_inf      = next(model_inf.parameters()).device

    # find one val image to test
    data_root = find_data_root()
    demo_img  = next(Path(data_root).rglob("*.tif"), None) or                 next(Path(data_root).rglob("*.png"), None)
    if demo_img:
        result = predict_image(model_inf, str(demo_img), cfg_inf, device_inf, tta=False)
        print(f"\n\U0001f9ea  Demo prediction on: {demo_img.name}")
        print(f"   Predicted class : {result['class']}")
        print(f"   Confidence      : {result['confidence']*100:.1f}%")
        for cls, prob in result["probabilities"].items():
            bar = "\u2588" * int(prob * 30)
            print(f"   {cls:10s}  {prob*100:5.1f}%  {bar}")
else:
    print("\u2139\ufe0f  Weights not found yet \u2014 run training first, then re-run this cell.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  SAVE TO KAGGLE OUTPUT  — run any time during or after training
#
#  Kaggle persists files in /kaggle/working/ when you "Save Version" (commit).
#  Files in subdirectories like /kaggle/working/checkpoints/ are NOT committed
#  automatically — this cell copies your best weights to the top-level Output.
#
#  After running: Kaggle notebook → right panel → Output → download .pth files
# ═══════════════════════════════════════════════════════════════════════════════
import shutil
from pathlib import Path

cfg       = Config()
ckpt_dir  = Path(cfg.CHECKPOINT_DIR)
output    = Path("/kaggle/working")

copied = []

# 1. All unique per-epoch best-weight backups
for f in sorted(ckpt_dir.glob("model_best_ep*.pth")):
    dest = output / f.name
    shutil.copy2(f, dest)
    copied.append((dest, dest.stat().st_size / 1e6))

# 2. The portable weights-only file
wbest = ckpt_dir / "model_weights_best.pth"
if wbest.exists():
    dest = output / wbest.name
    shutil.copy2(wbest, dest)
    copied.append((dest, dest.stat().st_size / 1e6))

# 3. Full resume checkpoint
cbest = ckpt_dir / "checkpoint_best.pth"
if cbest.exists():
    dest = output / cbest.name
    shutil.copy2(cbest, dest)
    copied.append((dest, dest.stat().st_size / 1e6))

if copied:
    print("Files saved to Kaggle Output:\n")
    for p, mb in copied:
        print(f"  ✅  {p.name:<40}  {mb:.1f} MB")
    print("\n  → Kaggle notebook → Output tab → download")
    print("  → Load with: load_model_for_inference(weights_path='<filename>')")
else:
    print("  ⚠  No checkpoint files found yet. Train at least one epoch first.")
